In [1]:
import joblib

model = joblib.load("../models/model.pkl")


In [2]:
valid = joblib.load("../data/final_dataset/processed/valid.pkl")
pred = joblib.load("../data/final_dataset/processed/val_data_pred.pkl")

In [3]:
import pandas as pd
valid = valid.copy()
valid['prediction'] = pred

valid["segment"] = pd.qcut(
    valid["prediction"],
    q=[0, 0.5, 0.8, 0.95, 1.0],
    labels=[
        "Low-Engagement",
        "At-Risk",
        "Growth",
        "High-Value"
    ]
)

In [4]:
valid.groupby("segment")["prediction"].agg(["count", "mean", "sum"])

/tmp/ipykernel_67277/2956156373.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  valid.groupby("segment")["prediction"].agg(["count", "mean", "sum"])


,count,mean,sum
segment,,,
Low-Engagement,21244,3.071419e+05,6.524922e+09
At-Risk,12746,2.481714e+06,3.163193e+10
Growth,6373,9.740026e+06,6.207319e+10
High-Value,2125,9.027460e+07,1.918335e+11


In [5]:
valid.head()

,client_id,year_month,total_amount,transaction_count,avg_transaction,target_next_month,amount_lag_1,count_lag_1,amount_lag_3,count_lag_3,amount_lag_6,count_lag_6,amount_roll_3m,amount_roll_6m,prediction,segment
14,000afc90f17bbfcfe559432c975a1f7b65fd8c27ba222c...,2022-09,419431.10,9,46603.453,465537.50,565458.00,4.0,475455.97,11.0,1445709.90,9.0,1.015168e+06,758472.041667,656694.006343,Low-Engagement
15,000afc90f17bbfcfe559432c975a1f7b65fd8c27ba222c...,2022-10,465537.50,4,116384.375,323532.06,419431.10,9.0,2060616.10,10.0,331259.56,16.0,4.834755e+05,780851.697917,395907.252150,Low-Engagement
16,000afc90f17bbfcfe559432c975a1f7b65fd8c27ba222c...,2022-11,323532.06,5,64706.414,229300.53,465537.50,4.0,565458.00,4.0,698611.50,10.0,4.028336e+05,718338.458333,380384.558158,Low-Engagement
31,000e047a31e50ba35f71c81962b4eb0b9a2d6080cf23a1...,2022-09,582652.56,22,26484.207,896475.06,654413.56,19.0,531542.70,13.0,577238.60,25.0,5.901361e+05,569287.187500,621613.505662,Low-Engagement
32,000e047a31e50ba35f71c81962b4eb0b9a2d6080cf23a1...,2022-10,896475.06,21,42689.290,781291.90,582652.56,22.0,533342.25,17.0,558868.60,11.0,7.111804e+05,625554.927083,663932.498203,Low-Engagement


In [6]:
growth = valid[valid["segment"] == "Growth"]

avg_check = growth["prediction"].mean()
n_clients = growth["client_id"].nunique()

discount = 0.10
expected_uplift = avg_check * discount * n_clients
expected_uplift

np.float64(2792465491.293584)

Модель прогнозирует суммарную транзакционную активность клиента в следующем месяце.  
Клиенты сегментируются на 4 группы по ожидаемой ценности, что позволяет перераспределять маркетинговый бюджет.  